# Imports

In [182]:
import os

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr, pointbiserialr
from scipy.stats import f_oneway, kruskal

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.metrics import r2_score, roc_curve, auc

from autogluon.tabular import TabularDataset, TabularPredictor
#from autogluon.multimodal import MultiModalPredictor
import xgboost as xgb

plt.style.use('default')
sns.set_palette("husl")
SEED = 43
np.random.seed(SEED)

VAL_TILE_FRACTION = 0.4
TILE_SIZE_DEG = 0.01  # approx 1km


SCORING = {
    'r2': 'r2',
    'mae': 'neg_mean_absolute_error',
    'mse': 'neg_mean_squared_error'
}
GRID_SEARCH_KWARGS = dict(
    cv=10,
    scoring=SCORING,
    refit='r2',
    n_jobs=-1,
    return_train_score=True
)
LINEAR_PARAM_GRID = {
    'model__fit_intercept': [True, False],
    'model__positive': [False, True]
}
TREE_PARAM_GRID = {
    'model__max_depth': [None, 10, 20],
    'model__min_samples_split': [1, 100],
    'model__min_samples_leaf': [2, 100],
    'model__max_features': [None, 'sqrt', 'log2']
}
RF_PARAM_GRID = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [None, 10, 20],
    'model__min_samples_split': [1, 100],
    'model__min_samples_leaf': [2, 100],
    'model__max_features': [None, 'sqrt', 'log2']
}
MLP_PARAM_GRID = {
    'model__hidden_layer_sizes': [(100,), (100, 50), (50, 25)],
    'model__activation': ['relu', 'tanh'],
    'model__alpha': [0.0001, 0.001, 0.01],
    'model__learning_rate_init': [0.001, 0.01]
}
XGB_PARAM_GRID = {
    'model__learning_rate': [0.01, 0.1, 0.2],
    'model__n_estimators': [50, 100, 200]
}

def evaluate_regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    errors = y_true - y_pred

    bin_size = 0.1
    bins = np.arange(0.0, 1.0 + bin_size, bin_size)
    bin_ids = np.digitize(y_pred, bins, right=False)

    ece = 0.0
    N = len(y_pred)

    for i in range(1, len(bins)):
        mask = bin_ids == i
        if not np.any(mask):
            continue

        bin_center = (bins[i - 1] + bins[i]) / 2.0
        median_obs = np.median(y_true[mask])
        ece += (mask.sum() / N) * abs(median_obs - bin_center)
        
    return {
        'r2': r2_score(y_true, y_pred),
        'mae': float(np.mean(np.abs(errors))),
        'mse': float(np.mean(errors ** 2)),
        "p95": np.quantile(np.abs(errors), 0.95),
        'ece': float(ece),              # Expected calibration error
    }


def record_regression_results(results, target, model_name, y_true, y_pred, split='test'):
    metrics = evaluate_regression_metrics(y_true, y_pred)
    split_label = split.capitalize()
    print(f"{split_label} R2 score {metrics['r2']:.4f}")
    print(f"{split_label} MAE score {metrics['mae']:.4f}")
    print(f"{split_label} MSE score {metrics['mse']:.4f}")
    print(f"{split_label} P95 score {metrics['p95']:.4f}")
    print(f"{split_label} ECE score {metrics['ece']:.4f}")
    results.append({
        'target': target,
        'model': model_name,
        f'{split}_r2': metrics['r2'],
        f'{split}_mae': metrics['mae'],
        #f'{split}_mse': metrics['mse'],
        #f'{split}_p95': metrics['p95'],
        f'{split}_ece': metrics['ece'],
    })
    return metrics


# Load data

In [183]:
model_det = "yolo"
data_path_file = f"{model_det}_metafeatures.csv"
data = pd.read_csv("../data/" + data_path_file, index_col=0)

print(data)

      country time_of_day        lat       long       road_type  \
1          PL         day  52.249511  21.043137            city   
6          PL         day  52.239332  21.030383            city   
39         PL         day  52.234241  21.003985            city   
49         PL         day  52.224666  21.071192            city   
52         PL    twilight  52.231355  21.054809            city   
...       ...         ...        ...        ...             ...   
99904      PL         day  54.353453  18.645450  arterial-urban   
99939      DE       night  50.116028   8.622851         highway   
99941      DE         day  53.057472   8.958626  arterial-urban   
99984      IT       night  41.935093  12.599486   smaller-rural   
99997      SE    twilight  59.103593  12.789612  arterial-rural   

      road_condition              weather  solar_angle_elevation  month  hour  \
1             normal    partly-cloudy-day              51.723833      4    10   
6             normal            c

In [184]:
data = data.dropna()
num_rows, num_cols = data.shape
print(f"After dropping missing values: {num_rows:,} rows and {num_cols} columns")

After dropping missing values: 9,655 rows and 42 columns


In [185]:
iou = data["iou"]
lrp = data["lrp"]
conf = data["mean_conf"]
image_paths_dict = {int(img_pth.split("_")[0]):f"../data/zod_yolo/images/test/{img_pth}" for img_pth in os.listdir("../data/zod_yolo/images/test") if img_pth.endswith(".jpg")}
img_path = pd.DataFrame.from_dict(image_paths_dict, orient='index', columns=["Images"])

data = data.drop(columns=["iou", "lrp"])
data_indices = data.index.to_numpy()

In [186]:
all_columns = data.columns    
all_columns = all_columns.drop(["weather_code", "cloud_cover_low", "cloud_cover_mid"])

print(f"Number of rows: {data.shape[0]}")
print(f"Columns: {all_columns.sort_values()}")
print(f"Number of columns: {len(all_columns)}")

Number of rows: 9655
Columns: Index(['brightness', 'camera_distance_from_ground', 'camera_offset',
       'camera_pitch_angle', 'cloud_cover', 'complexity', 'contrast',
       'country', 'distortion_magnitude', 'field_view_horizontal',
       'forward_acceleration', 'forward_velocity', 'hour', 'laplacian', 'lat',
       'lateral_acceleration', 'lateral_velocity', 'long', 'max_conf',
       'mean_conf', 'min_conf', 'month', 'noisiness', 'num_detections',
       'quality', 'rain', 'relative_humidity_2m', 'road_condition',
       'road_type', 'sharpness', 'snowfall', 'solar_angle_elevation',
       'std_conf', 'temperature_2m', 'time_of_day', 'weather',
       'wind_speed_10m'],
      dtype='object')
Number of columns: 37


In [187]:
numeric_columns = ['solar_angle_elevation', 'temperature_2m', 'relative_humidity_2m', 'rain', 'snowfall', 'cloud_cover', 'wind_speed_10m', 'forward_acceleration', 'lateral_acceleration', 'forward_velocity', 'lateral_velocity', 'field_view_horizontal', 'camera_distance_from_ground', 'camera_pitch_angle', 'distortion_magnitude', 'camera_offset', 'laplacian', 'brightness', 'contrast', 'sharpness', 'noisiness', 'quality', 'complexity']
numeric_columns = numeric_columns + ['mean_conf', "std_conf", "max_conf", "min_conf", "num_detections"]

categorical_columns = ["country", "time_of_day", "road_type", "road_condition", "weather"]
other = ["hour", "month", "lat", "long"]
print(sorted(numeric_columns + categorical_columns + other))
assert len(all_columns) == (len(categorical_columns) + len(numeric_columns) + len(other)), "Columns not match"

['brightness', 'camera_distance_from_ground', 'camera_offset', 'camera_pitch_angle', 'cloud_cover', 'complexity', 'contrast', 'country', 'distortion_magnitude', 'field_view_horizontal', 'forward_acceleration', 'forward_velocity', 'hour', 'laplacian', 'lat', 'lateral_acceleration', 'lateral_velocity', 'long', 'max_conf', 'mean_conf', 'min_conf', 'month', 'noisiness', 'num_detections', 'quality', 'rain', 'relative_humidity_2m', 'road_condition', 'road_type', 'sharpness', 'snowfall', 'solar_angle_elevation', 'std_conf', 'temperature_2m', 'time_of_day', 'weather', 'wind_speed_10m']


# Assessor Tests

Split data into train 60% and validation 40%. To prevent leakage, the data is binned in buckets by lat/long in 1km. The split is done by buckets ensuring no leakage. 

In [188]:
#Val split
def tile_id(lat, lon, size_deg):
    lat_bin = np.floor(lat / size_deg) * size_deg
    lon_bin = np.floor(lon / size_deg) * size_deg
    return f'{lat_bin:.4f}_{lon_bin:.4f}'

In [189]:
## Make split of train into train and val by tiles
ids = data.index.tolist()
coords = {frame_id: {"lat": data.loc[frame_id, "lat"], "lon": data.loc[frame_id, "long"]} for frame_id in ids}
df_coords = pd.DataFrame.from_dict(coords, orient='index')
df_coords['tile_id'] = [tile_id(lat, lon, TILE_SIZE_DEG) for lat, lon in zip(df_coords['lat'], df_coords['lon'])]

unique_tiles = sorted(set(df_coords['tile_id'].dropna().unique()))
rng = np.random.default_rng(SEED)
val_tile_count = int(len(unique_tiles) * VAL_TILE_FRACTION)
val_tiles = set(rng.choice(unique_tiles, size=val_tile_count, replace=False))
df_coords['split'] = np.where(df_coords['tile_id'].isin(val_tiles), 'val', 'train')
test_idx = df_coords[df_coords['split'] == 'val'].index.tolist()
train_idx = df_coords[df_coords['split'] == 'train'].index.tolist()
print(f"Total train frames: {len(train_idx)}, val frames: {len(test_idx)}")

Total train frames: 5518, val frames: 4137


In [190]:
X_train = data.loc[train_idx]
y_train = iou.loc[train_idx]
y_train_lrp = lrp.loc[train_idx]
conf_train = conf.loc[train_idx]
conf_train = pd.DataFrame(conf_train)

X_test = data.loc[test_idx]
y_test = iou.loc[test_idx]
y_test_lrp = lrp.loc[test_idx]
conf_test = conf.loc[test_idx]
conf_test = pd.DataFrame(conf_test)

print("X:", len(X_train), len(X_test))
print("y:", len(y_train), len(y_test))
print("Conf:", len(conf_train), len(conf_test))
print(X_train.columns)

X: 5518 4137
y: 5518 4137
Conf: 5518 4137
Index(['country', 'time_of_day', 'lat', 'long', 'road_type', 'road_condition',
       'weather', 'solar_angle_elevation', 'month', 'hour',
       'forward_acceleration', 'lateral_acceleration', 'forward_velocity',
       'lateral_velocity', 'field_view_horizontal',
       'camera_distance_from_ground', 'camera_pitch_angle',
       'distortion_magnitude', 'camera_offset', 'laplacian', 'quality',
       'brightness', 'noisiness', 'sharpness', 'contrast', 'complexity',
       'temperature_2m', 'relative_humidity_2m', 'rain', 'snowfall',
       'cloud_cover', 'cloud_cover_low', 'cloud_cover_mid', 'wind_speed_10m',
       'weather_code', 'num_detections', 'max_conf', 'min_conf', 'mean_conf',
       'std_conf'],
      dtype='object')


In [191]:
model_results = []

### IoU

#### Baseline

Random Prediction. Mean of metric over training set.

In [192]:
mean_iou_train = np.mean(y_train)
iou_pred_test = np.full_like(y_test, mean_iou_train)
record_regression_results(model_results, 'IoU', 'Constant Mean Predictor', y_test, iou_pred_test)


Test R2 score -0.0064
Test MAE score 0.1535
Test MSE score 0.0384
Test P95 score 0.4406
Test ECE score 0.0237


{'r2': -0.006387059165034126,
 'mae': 0.15345011285057936,
 'mse': 0.03843070450386507,
 'p95': 0.4406463144463975,
 'ece': 0.023650633587556813}

#### Linear Reg on Conf

In [193]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['mean_conf']),
    ],
    remainder='passthrough'
)
linear_reg_conf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])
linear_reg_conf_grid = GridSearchCV(
    linear_reg_conf_pipeline,
    param_grid=LINEAR_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
linear_reg_conf_grid.fit(conf_train, y_train)

best_idx = linear_reg_conf_grid.best_index_
mean_train_r2 = linear_reg_conf_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = linear_reg_conf_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -linear_reg_conf_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -linear_reg_conf_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -linear_reg_conf_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -linear_reg_conf_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {linear_reg_conf_grid.best_params_}")
best_linear_reg_conf = linear_reg_conf_grid.best_estimator_


Mean CV train r2 score 0.3696
Mean CV test r2 score 0.2929
Mean CV train MAE score 0.1206
Mean CV test MAE score 0.1214
Mean CV train MSE score 0.0243
Mean CV test MSE score 0.0246
Best params: {'model__fit_intercept': True, 'model__positive': True}


In [194]:
y_pred_iou_baseline = best_linear_reg_conf.predict(conf_test)
record_regression_results(model_results, 'IoU', 'Univariate Linear Regression (Confidence)', y_test, y_pred_iou_baseline)


Test R2 score 0.3885
Test MAE score 0.1181
Test MSE score 0.0234
Test P95 score 0.3101
Test ECE score 0.0122


{'r2': 0.38852995045885996,
 'mae': 0.11810461387048779,
 'mse': 0.0233500864035114,
 'p95': 0.3100963387616801,
 'ece': 0.012189133650501907}

#### Linear Regression

Train models with cv and then test.

In [195]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
linear_reg_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])
linear_reg_grid = GridSearchCV(
    linear_reg_pipeline,
    param_grid=LINEAR_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
linear_reg_grid.fit(X_train, y_train)

best_idx = linear_reg_grid.best_index_
mean_train_r2 = linear_reg_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = linear_reg_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -linear_reg_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -linear_reg_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -linear_reg_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -linear_reg_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {linear_reg_grid.best_params_}")
best_linear_reg = linear_reg_grid.best_estimator_

Mean CV train r2 score 0.4665
Mean CV test r2 score 0.3857
Mean CV train MAE score 0.1111
Mean CV test MAE score 0.1141
Mean CV train MSE score 0.0205
Mean CV test MSE score 0.0215
Best params: {'model__fit_intercept': True, 'model__positive': False}


In [196]:
y_pred_iou_lr = best_linear_reg.predict(X_test)
record_regression_results(model_results, 'IoU', 'Linear Regression', y_test, y_pred_iou_lr)


Test R2 score 0.4767
Test MAE score 0.1090
Test MSE score 0.0200
Test P95 score 0.2903
Test ECE score 0.0118


{'r2': 0.4766620513629167,
 'mae': 0.1090383326279859,
 'mse': 0.01998460321659653,
 'p95': 0.29025686207424284,
 'ece': 0.011830057238950554}

#### Decision Trees

In [197]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
decision_tree_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', DecisionTreeRegressor(random_state=SEED))
])
decision_tree_grid = GridSearchCV(
    decision_tree_pipeline,
    param_grid=TREE_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
decision_tree_grid.fit(X_train, y_train)

best_idx = decision_tree_grid.best_index_
mean_train_r2 = decision_tree_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = decision_tree_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -decision_tree_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -decision_tree_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -decision_tree_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -decision_tree_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {decision_tree_grid.best_params_}")
best_decision_tree = decision_tree_grid.best_estimator_


Mean CV train r2 score 0.4736
Mean CV test r2 score 0.3631
Mean CV train MAE score 0.1088
Mean CV test MAE score 0.1153
Mean CV train MSE score 0.0203
Mean CV test MSE score 0.0226
Best params: {'model__max_depth': None, 'model__max_features': None, 'model__min_samples_leaf': 100, 'model__min_samples_split': 100}


/home/felix/miniconda3/envs/pred_ad/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:516: FitFailedWarning: 
180 fits failed out of a total of 360.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
180 fits failed with the following error:
Traceback (most recent call last):
  File "/home/felix/miniconda3/envs/pred_ad/lib/python3.11/site-packages/sklearn/model_selection/_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/home/felix/miniconda3/envs/pred_ad/lib/python3.11/site-packages/sklearn/base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/felix/miniconda3/envs/pred_ad/lib/py

In [198]:
y_pred_iou_dt = best_decision_tree.predict(X_test)
record_regression_results(model_results, 'IoU', 'Decision Tree', y_test, y_pred_iou_dt)


Test R2 score 0.4506
Test MAE score 0.1109
Test MSE score 0.0210
Test P95 score 0.2944
Test ECE score 0.0109


{'r2': 0.4506245000026423,
 'mae': 0.11086581023547348,
 'mse': 0.020978894064454922,
 'p95': 0.29442066015811613,
 'ece': 0.010919332691081616}

#### Random Forest

In [199]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
rand_forest_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=SEED, n_jobs=-1))
])
rand_forest_grid = GridSearchCV(
    rand_forest_pipeline,
    param_grid=RF_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
rand_forest_grid.fit(X_train, y_train)

best_idx = rand_forest_grid.best_index_
mean_train_r2 = rand_forest_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = rand_forest_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -rand_forest_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -rand_forest_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -rand_forest_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -rand_forest_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {rand_forest_grid.best_params_}")
best_rand_forest = rand_forest_grid.best_estimator_


/home/felix/miniconda3/envs/pred_ad/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:516: FitFailedWarning: 
360 fits failed out of a total of 720.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
360 fits failed with the following error:
Traceback (most recent call last):
  File "/home/felix/miniconda3/envs/pred_ad/lib/python3.11/site-packages/sklearn/model_selection/_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/home/felix/miniconda3/envs/pred_ad/lib/python3.11/site-packages/sklearn/base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/felix/miniconda3/envs/pred_ad/lib/py

Mean CV train r2 score 0.5699
Mean CV test r2 score 0.3860
Mean CV train MAE score 0.0987
Mean CV test MAE score 0.1125
Mean CV train MSE score 0.0166
Mean CV test MSE score 0.0217
Best params: {'model__max_depth': 20, 'model__max_features': None, 'model__min_samples_leaf': 2, 'model__min_samples_split': 100, 'model__n_estimators': 200}


In [200]:
y_pred_iou_rf = best_rand_forest.predict(X_test)
record_regression_results(model_results, 'IoU', 'Random Forest', y_test, y_pred_iou_rf)


Test R2 score 0.4869
Test MAE score 0.1068
Test MSE score 0.0196
Test P95 score 0.2857
Test ECE score 0.0146


{'r2': 0.48690160266090976,
 'mae': 0.10680901881882209,
 'mse': 0.019593587487010518,
 'p95': 0.28573999109264436,
 'ece': 0.014599255049945982}

#### MLP

In [201]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
mlp_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', MLPRegressor(max_iter=500, random_state=SEED))
])
mlp_grid = GridSearchCV(
    mlp_pipeline,
    param_grid=MLP_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
mlp_grid.fit(X_train, y_train)

best_idx = mlp_grid.best_index_
mean_train_r2 = mlp_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = mlp_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -mlp_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -mlp_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -mlp_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -mlp_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {mlp_grid.best_params_}")
best_mlp = mlp_grid.best_estimator_


Mean CV train r2 score 0.4872
Mean CV test r2 score 0.3024
Mean CV train MAE score 0.1092
Mean CV test MAE score 0.1208
Mean CV train MSE score 0.0198
Mean CV test MSE score 0.0247
Best params: {'model__activation': 'relu', 'model__alpha': 0.01, 'model__hidden_layer_sizes': (100, 50), 'model__learning_rate_init': 0.01}


In [202]:
y_pred_iou_mlp = best_mlp.predict(X_test)
record_regression_results(model_results, 'IoU', 'MLP', y_test, y_pred_iou_mlp)


Test R2 score 0.4486
Test MAE score 0.1099
Test MSE score 0.0211
Test P95 score 0.3008
Test ECE score 0.0077


{'r2': 0.44858426716617117,
 'mae': 0.10989121849370553,
 'mse': 0.02105680403412659,
 'p95': 0.30076325749794136,
 'ece': 0.007714882608393278}

#### XGBoost

In [203]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
xgb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', xgb.XGBRegressor())
])
xgb_grid = GridSearchCV(
    xgb_pipeline,
    param_grid=XGB_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
xgb_grid.fit(X_train, y_train)


best_idx = xgb_grid.best_index_
mean_train_r2 = xgb_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = xgb_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -xgb_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -xgb_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -xgb_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -xgb_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {xgb_grid.best_params_}")
best_xgb = xgb_grid.best_estimator_

Mean CV train r2 score 0.7277
Mean CV test r2 score 0.3907
Mean CV train MAE score 0.0804
Mean CV test MAE score 0.1115
Mean CV train MSE score 0.0105
Mean CV test MSE score 0.0215
Best params: {'model__learning_rate': 0.1, 'model__n_estimators': 50}


In [204]:
y_pred_iou_xgb = best_xgb.predict(X_test)
record_regression_results(model_results, 'IoU', 'XGBoost', y_test, y_pred_iou_xgb)


Test R2 score 0.4853
Test MAE score 0.1064
Test MSE score 0.0197
Test P95 score 0.2855
Test ECE score 0.0088


{'r2': 0.4852756792516607,
 'mae': 0.10635852342581197,
 'mse': 0.019655676304148738,
 'p95': 0.2855141401290893,
 'ece': 0.00879654258009675}

#### Autogluon

##### Tabluar

In [205]:
train = pd.merge(X_train, y_train, left_index=True, right_index=True)

In [206]:
autoglue_model = TabularPredictor(label="iou", problem_type="regression", eval_metric="r2", path="../models/assessors/autoglue_tab/iou/")
autoglue_predictor = autoglue_model.fit(train)


Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.11.14
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #100-Ubuntu SMP PREEMPT_DYNAMIC Tue Jan 13 16:40:06 UTC 2026
CPU Count:          32
Pytorch Version:    2.9.1+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 23.55/23.55 GB
Total GPU Memory:   Free: 23.55 GB, Allocated: 0.00 GB, Total: 23.55 GB
GPU Count:          1
Memory Avail:       4.75 GB / 30.46 GB (15.6%)
Disk Space Avail:   1902.37 GB / 3542.28 GB (53.7%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='extreme'  : New in v1.5: The state-of-the-art for tabular data. Massively better than 'best' on datasets <100000 samples by using new 

In [207]:
autoglue_predictor.fit_summary(verbosity=2)

*** Summary of fit() ***
Estimated performance of each model:
                 model  score_val eval_metric  pred_time_val  fit_time  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0  WeightedEnsemble_L2   0.455792          r2       0.037421  2.540382                0.000203           0.015540            2       True          9
1              XGBoost   0.446091          r2       0.009353  0.343369                0.009353           0.343369            1       True          6
2             CatBoost   0.445285          r2       0.001859  0.686280                0.001859           0.686280            1       True          4
3           LightGBMXT   0.442331          r2       0.001159  0.489171                0.001159           0.489171            1       True          1
4      RandomForestMSE   0.436876          r2       0.024846  1.006022                0.024846           1.006022            1       True          3
5        ExtraTreesMSE   0.423553          r

/home/felix/miniconda3/envs/pred_ad/lib/python3.11/site-packages/autogluon/core/utils/plots.py:169: UserWarning: AutoGluon summary plots cannot be created because bokeh is not installed. To see plots, please do: "pip install bokeh==2.0.1"
  warnings.warn('AutoGluon summary plots cannot be created because bokeh is not installed. To see plots, please do: "pip install bokeh==2.0.1"')


{'model_types': {'LightGBMXT': 'LGBModel',
  'LightGBM': 'LGBModel',
  'RandomForestMSE': 'RFModel',
  'CatBoost': 'CatBoostModel',
  'ExtraTreesMSE': 'XTModel',
  'XGBoost': 'XGBoostModel',
  'NeuralNetTorch': 'TabularNeuralNetTorchModel',
  'LightGBMLarge': 'LGBModel',
  'WeightedEnsemble_L2': 'WeightedEnsembleModel'},
 'model_performance': {'LightGBMXT': 0.44233105144830254,
  'LightGBM': 0.42067312457935335,
  'RandomForestMSE': 0.4368764298099044,
  'CatBoost': 0.44528485739543566,
  'ExtraTreesMSE': 0.42355260459620925,
  'XGBoost': 0.4460910794847005,
  'NeuralNetTorch': 0.3914869413749982,
  'LightGBMLarge': 0.38446280786747455,
  'WeightedEnsemble_L2': 0.4557921437575877},
 'model_best': 'WeightedEnsemble_L2',
 'model_paths': {'LightGBMXT': ['LightGBMXT'],
  'LightGBM': ['LightGBM'],
  'RandomForestMSE': ['RandomForestMSE'],
  'CatBoost': ['CatBoost'],
  'ExtraTreesMSE': ['ExtraTreesMSE'],
  'XGBoost': ['XGBoost'],
  'NeuralNetTorch': ['NeuralNetTorch'],
  'LightGBMLarge': ['L

In [208]:
y_pred_iou_autg = autoglue_predictor.predict(X_test)
record_regression_results(model_results, 'IoU', 'Autogluon_Tab', y_test, y_pred_iou_autg)


Test R2 score 0.4957
Test MAE score 0.1066
Test MSE score 0.0193
Test P95 score 0.2835
Test ECE score 0.0121


{'r2': 0.4956502474876999,
 'mae': 0.1065972921956623,
 'mse': 0.019259504709329942,
 'p95': 0.28345192074775694,
 'ece': 0.012099689639310919}

##### Images

In [209]:
X_train_img = pd.merge(X_train, img_path, left_index=True, right_index=True)
X_test_img = pd.merge(X_test, img_path, how="left", right_index=True, left_index=True)
train_img = pd.merge(X_train_img, y_train, left_index=True, right_index=True)

In [210]:
#autoglue_model_img = MultiModalPredictor(label="iou", problem_type="regression", eval_metric="r2", path="../models/assessors/autoglue_tab/iou/img/")
hyperparams = {
    "env.strategy": "ddp_notebook",
    "env.num_gpus": 1,
    "env.num_workers": 0,
    #"optim.max_epochs": 15,
    #"optim.patience": 3,
    #"optim.learning_rate": 1e-4,
    #"optim.weight_decay": 1e-4,
    #"optim.warmup_steps": 500,
    #"env.per_gpu_batch_size": 8,
    #"env.batch_size": 128,
    #"model.timm_image.checkpoint_name": "convnext_base",
}
#autoglue_predictor_img = autoglue_model_img.fit(train_img, hyperparameters=hyperparams)

In [211]:
#autoglue_predictor_img.fit_summary(verbosity=2)

In [212]:
#y_pred = autoglue_predictor_img.predict(X_test_img)
#record_regression_results(model_results, 'IoU', 'Autogluon_Mult', y_test, y_pred)


### LRP


#### Baseline

Predict Only the Mean


In [213]:
mean_lrp_train = np.mean(y_train_lrp)
lrp_pred_test = np.full_like(y_test_lrp, mean_lrp_train)
record_regression_results(model_results, 'LRP', 'Constant Mean Predictor', y_test_lrp, lrp_pred_test)


Test R2 score -0.0059
Test MAE score 0.1373
Test MSE score 0.0308
Test P95 score 0.3379
Test ECE score 0.0403


{'r2': -0.00590215302083541,
 'mae': 0.13728463526477125,
 'mse': 0.030806930297524026,
 'p95': 0.33788661736267206,
 'ece': 0.040282642841339145}

#### Linear Reg on Conf

In [214]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['mean_conf']),
    ],
    remainder='passthrough'
)
linear_reg_conf_lrp_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])
linear_reg_conf_lrp_grid = GridSearchCV(
    linear_reg_conf_lrp_pipeline,
    param_grid=LINEAR_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
linear_reg_conf_lrp_grid.fit(conf_train, y_train_lrp)

best_idx = linear_reg_conf_lrp_grid.best_index_
mean_train_r2 = linear_reg_conf_lrp_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = linear_reg_conf_lrp_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -linear_reg_conf_lrp_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -linear_reg_conf_lrp_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -linear_reg_conf_lrp_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -linear_reg_conf_lrp_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {linear_reg_conf_lrp_grid.best_params_}")
best_linear_reg_conf_lrp = linear_reg_conf_lrp_grid.best_estimator_


Mean CV train r2 score 0.4453
Mean CV test r2 score 0.3909
Mean CV train MAE score 0.1014
Mean CV test MAE score 0.1020
Mean CV train MSE score 0.0175
Mean CV test MSE score 0.0177
Best params: {'model__fit_intercept': True, 'model__positive': False}


In [215]:
y_pred_lrp_baseline = best_linear_reg_conf_lrp.predict(conf_test)
record_regression_results(model_results, 'LRP', 'Univariate Linear Regression (Confidence)', y_test_lrp, y_pred_lrp_baseline)


Test R2 score 0.4597
Test MAE score 0.0989
Test MSE score 0.0165
Test P95 score 0.2683
Test ECE score 0.0175


{'r2': 0.45968258033167597,
 'mae': 0.09890136490864726,
 'mse': 0.016547853124950326,
 'p95': 0.2682641402286816,
 'ece': 0.017501171748362118}

#### Linear Regression


Train models with cv and then test.


In [216]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
linear_reg_lrp_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])
linear_reg_lrp_grid = GridSearchCV(
    linear_reg_lrp_pipeline,
    param_grid=LINEAR_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
linear_reg_lrp_grid.fit(X_train, y_train_lrp)

best_idx = linear_reg_lrp_grid.best_index_
mean_train_r2 = linear_reg_lrp_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = linear_reg_lrp_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -linear_reg_lrp_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -linear_reg_lrp_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -linear_reg_lrp_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -linear_reg_lrp_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {linear_reg_lrp_grid.best_params_}")
best_linear_reg_lrp = linear_reg_lrp_grid.best_estimator_


Mean CV train r2 score 0.5172
Mean CV test r2 score 0.4553
Mean CV train MAE score 0.0933
Mean CV test MAE score 0.0956
Mean CV train MSE score 0.0152
Mean CV test MSE score 0.0159
Best params: {'model__fit_intercept': True, 'model__positive': False}


In [217]:
y_pred_lrp_lr = best_linear_reg_lrp.predict(X_test)
record_regression_results(model_results, 'LRP', 'Linear Regression', y_test_lrp, y_pred_lrp_lr)


Test R2 score 0.5189
Test MAE score 0.0915
Test MSE score 0.0147
Test P95 score 0.2607
Test ECE score 0.0150


{'r2': 0.5189249396979194,
 'mae': 0.09149742137328941,
 'mse': 0.014733486558405232,
 'p95': 0.2606546673727428,
 'ece': 0.015017143798848472}

#### Decision Trees


In [218]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
decision_tree_lrp_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', DecisionTreeRegressor(random_state=SEED))
])
decision_tree_lrp_grid = GridSearchCV(
    decision_tree_lrp_pipeline,
    param_grid=TREE_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
decision_tree_lrp_grid.fit(X_train, y_train_lrp)

best_idx = decision_tree_lrp_grid.best_index_
mean_train_r2 = decision_tree_lrp_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = decision_tree_lrp_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -decision_tree_lrp_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -decision_tree_lrp_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -decision_tree_lrp_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -decision_tree_lrp_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {decision_tree_lrp_grid.best_params_}")
best_decision_tree_lrp = decision_tree_lrp_grid.best_estimator_


Mean CV train r2 score 0.5342
Mean CV test r2 score 0.4376
Mean CV train MAE score 0.0902
Mean CV test MAE score 0.0956
Mean CV train MSE score 0.0147
Mean CV test MSE score 0.0165
Best params: {'model__max_depth': None, 'model__max_features': None, 'model__min_samples_leaf': 100, 'model__min_samples_split': 100}


/home/felix/miniconda3/envs/pred_ad/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:516: FitFailedWarning: 
180 fits failed out of a total of 360.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
180 fits failed with the following error:
Traceback (most recent call last):
  File "/home/felix/miniconda3/envs/pred_ad/lib/python3.11/site-packages/sklearn/model_selection/_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/home/felix/miniconda3/envs/pred_ad/lib/python3.11/site-packages/sklearn/base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/felix/miniconda3/envs/pred_ad/lib/py

In [219]:
y_pred_lrp_dt = best_decision_tree_lrp.predict(X_test)
record_regression_results(model_results, 'LRP', 'Decision Tree', y_test_lrp, y_pred_lrp_dt)


Test R2 score 0.5172
Test MAE score 0.0907
Test MSE score 0.0148
Test P95 score 0.2536
Test ECE score 0.0190


{'r2': 0.5171691749449157,
 'mae': 0.09069766353348649,
 'mse': 0.014787258908133483,
 'p95': 0.25360244429806716,
 'ece': 0.018981552458282064}

#### Random Forest


In [220]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
rand_forest_lrp_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=SEED, n_jobs=-1))
])
rand_forest_lrp_grid = GridSearchCV(
    rand_forest_lrp_pipeline,
    param_grid=RF_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
rand_forest_lrp_grid.fit(X_train, y_train_lrp)

best_idx = rand_forest_lrp_grid.best_index_
mean_train_r2 = rand_forest_lrp_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = rand_forest_lrp_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -rand_forest_lrp_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -rand_forest_lrp_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -rand_forest_lrp_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -rand_forest_lrp_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {rand_forest_lrp_grid.best_params_}")
best_rand_forest_lrp = rand_forest_lrp_grid.best_estimator_


/home/felix/miniconda3/envs/pred_ad/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:516: FitFailedWarning: 
360 fits failed out of a total of 720.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
360 fits failed with the following error:
Traceback (most recent call last):
  File "/home/felix/miniconda3/envs/pred_ad/lib/python3.11/site-packages/sklearn/model_selection/_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/home/felix/miniconda3/envs/pred_ad/lib/python3.11/site-packages/sklearn/base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/felix/miniconda3/envs/pred_ad/lib/py

Mean CV train r2 score 0.6288
Mean CV test r2 score 0.4725
Mean CV train MAE score 0.0808
Mean CV test MAE score 0.0925
Mean CV train MSE score 0.0117
Mean CV test MSE score 0.0154
Best params: {'model__max_depth': None, 'model__max_features': None, 'model__min_samples_leaf': 2, 'model__min_samples_split': 100, 'model__n_estimators': 200}


In [221]:
y_pred_lrp_rf = best_rand_forest_lrp.predict(X_test)
record_regression_results(model_results, 'LRP', 'Random Forest', y_test_lrp, y_pred_lrp_rf)


Test R2 score 0.5493
Test MAE score 0.0874
Test MSE score 0.0138
Test P95 score 0.2468
Test ECE score 0.0123


{'r2': 0.5492845256503615,
 'mae': 0.0874026490951749,
 'mse': 0.013803688719231905,
 'p95': 0.24675095855439774,
 'ece': 0.012270821253112917}

#### MLP


In [222]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
mlp_lrp_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', MLPRegressor(max_iter=500, random_state=SEED))
])
mlp_lrp_grid = GridSearchCV(
    mlp_lrp_pipeline,
    param_grid=MLP_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
mlp_lrp_grid.fit(X_train, y_train_lrp)

best_idx = mlp_lrp_grid.best_index_
mean_train_r2 = mlp_lrp_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = mlp_lrp_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -mlp_lrp_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -mlp_lrp_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -mlp_lrp_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -mlp_lrp_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {mlp_lrp_grid.best_params_}")
best_mlp_lrp = mlp_lrp_grid.best_estimator_


Mean CV train r2 score 0.5660
Mean CV test r2 score 0.3976
Mean CV train MAE score 0.0895
Mean CV test MAE score 0.1001
Mean CV train MSE score 0.0137
Mean CV test MSE score 0.0176
Best params: {'model__activation': 'tanh', 'model__alpha': 0.01, 'model__hidden_layer_sizes': (100, 50), 'model__learning_rate_init': 0.01}


In [223]:
y_pred_lrp_mlp = best_mlp_lrp.predict(X_test)
record_regression_results(model_results, 'LRP', 'MLP', y_test_lrp, y_pred_lrp_mlp)


Test R2 score 0.5099
Test MAE score 0.0925
Test MSE score 0.0150
Test P95 score 0.2527
Test ECE score 0.0171


{'r2': 0.5099245665952548,
 'mae': 0.09249998065069377,
 'mse': 0.015009133514714867,
 'p95': 0.25271929556923006,
 'ece': 0.017126972314554775}

#### XGBoost

In [224]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
xgb_lrp_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', xgb.XGBRegressor())
])
xgb_lrp_grid = GridSearchCV(
    xgb_lrp_pipeline,
    param_grid=XGB_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
xgb_lrp_grid.fit(X_train, y_train_lrp)


best_idx = xgb_lrp_grid.best_index_
mean_train_r2 = xgb_lrp_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = xgb_lrp_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -xgb_lrp_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -xgb_lrp_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -xgb_lrp_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -xgb_lrp_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {xgb_lrp_grid.best_params_}")
best_xgb_lrp = xgb_lrp_grid.best_estimator_

Mean CV train r2 score 0.7826
Mean CV test r2 score 0.4744
Mean CV train MAE score 0.0643
Mean CV test MAE score 0.0918
Mean CV train MSE score 0.0069
Mean CV test MSE score 0.0155
Best params: {'model__learning_rate': 0.1, 'model__n_estimators': 50}


In [225]:
y_pred_lrp_xgb = best_xgb_lrp.predict(X_test)
record_regression_results(model_results, 'LRP', 'XGBoost', y_test_lrp, y_pred_lrp_xgb)


Test R2 score 0.5457
Test MAE score 0.0870
Test MSE score 0.0139
Test P95 score 0.2495
Test ECE score 0.0108


{'r2': 0.5457027025838126,
 'mae': 0.0870123422591486,
 'mse': 0.01391338624122036,
 'p95': 0.24945664107799523,
 'ece': 0.01082515185052776}

#### Autogluon

##### Tabluar

In [226]:
train_lrp = pd.merge(X_train, y_train_lrp, left_index=True, right_index=True)

In [227]:
autoglue_model_lrp = TabularPredictor(label="lrp", problem_type="regression", eval_metric="r2", path="../models/assessors/autoglue_tab/lrp/tab/")
autoglue_predictor_lrp = autoglue_model_lrp.fit(train_lrp)


Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.11.14
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #100-Ubuntu SMP PREEMPT_DYNAMIC Tue Jan 13 16:40:06 UTC 2026
CPU Count:          32
Pytorch Version:    2.9.1+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 23.55/23.55 GB
Total GPU Memory:   Free: 23.55 GB, Allocated: 0.00 GB, Total: 23.55 GB
GPU Count:          1
Memory Avail:       4.66 GB / 30.46 GB (15.3%)
Disk Space Avail:   1902.38 GB / 3542.28 GB (53.7%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='extreme'  : New in v1.5: The state-of-the-art for tabular data. Massively better than 'best' on datasets <100000 samples by using new 

In [228]:
autoglue_predictor_lrp.fit_summary(verbosity=2)

*** Summary of fit() ***
Estimated performance of each model:
                 model  score_val eval_metric  pred_time_val  fit_time  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0  WeightedEnsemble_L2   0.540574          r2       0.040746  2.177356                0.000208           0.014976            2       True          9
1             CatBoost   0.532990          r2       0.002025  0.748878                0.002025           0.748878            1       True          4
2      RandomForestMSE   0.530789          r2       0.025840  1.004192                0.025840           1.004192            1       True          3
3              XGBoost   0.525536          r2       0.012673  0.409310                0.012673           0.409310            1       True          6
4           LightGBMXT   0.525221          r2       0.001082  0.452535                0.001082           0.452535            1       True          1
5        LightGBMLarge   0.509997          r

/home/felix/miniconda3/envs/pred_ad/lib/python3.11/site-packages/autogluon/core/utils/plots.py:169: UserWarning: AutoGluon summary plots cannot be created because bokeh is not installed. To see plots, please do: "pip install bokeh==2.0.1"
  warnings.warn('AutoGluon summary plots cannot be created because bokeh is not installed. To see plots, please do: "pip install bokeh==2.0.1"')


{'model_types': {'LightGBMXT': 'LGBModel',
  'LightGBM': 'LGBModel',
  'RandomForestMSE': 'RFModel',
  'CatBoost': 'CatBoostModel',
  'ExtraTreesMSE': 'XTModel',
  'XGBoost': 'XGBoostModel',
  'NeuralNetTorch': 'TabularNeuralNetTorchModel',
  'LightGBMLarge': 'LGBModel',
  'WeightedEnsemble_L2': 'WeightedEnsembleModel'},
 'model_performance': {'LightGBMXT': 0.5252208683406507,
  'LightGBM': 0.5082151768207979,
  'RandomForestMSE': 0.5307890652076421,
  'CatBoost': 0.5329896386106401,
  'ExtraTreesMSE': 0.508949601939967,
  'XGBoost': 0.525535524623975,
  'NeuralNetTorch': 0.47282438241732894,
  'LightGBMLarge': 0.5099969812287397,
  'WeightedEnsemble_L2': 0.5405744705244764},
 'model_best': 'WeightedEnsemble_L2',
 'model_paths': {'LightGBMXT': ['LightGBMXT'],
  'LightGBM': ['LightGBM'],
  'RandomForestMSE': ['RandomForestMSE'],
  'CatBoost': ['CatBoost'],
  'ExtraTreesMSE': ['ExtraTreesMSE'],
  'XGBoost': ['XGBoost'],
  'NeuralNetTorch': ['NeuralNetTorch'],
  'LightGBMLarge': ['LightGB

In [229]:
y_pred_lrp_autg = autoglue_predictor_lrp.predict(X_test)
record_regression_results(model_results, 'LRP', 'Autogluon_Tab', y_test_lrp, y_pred_lrp_autg)


Test R2 score 0.5640
Test MAE score 0.0856
Test MSE score 0.0134
Test P95 score 0.2412
Test ECE score 0.0124


{'r2': 0.5640212022308088,
 'mae': 0.08556552589568203,
 'mse': 0.013352360757692498,
 'p95': 0.2412105917930602,
 'ece': 0.01240412412647588}

##### Images

In [230]:
X_train_img = pd.merge(X_train, img_path, left_index=True, right_index=True)
X_test_img = pd.merge(X_test, img_path, how="left", right_index=True, left_index=True)
train_lrp_img = pd.merge(X_train_img, y_train_lrp, left_index=True, right_index=True)

In [231]:
#autoglue_model_lrp_img = MultiModalPredictor(label="lrp", problem_type="regression", eval_metric="r2", path="../models/assessors/autoglue_tab/lrp/img/")
hyperparams = {
    "env.strategy": "ddp_notebook",
    "env.num_gpus": 1,
    "env.num_workers": 0,
    #"optim.max_epochs": 15,
    #"optim.patience": 3,
    #"optim.learning_rate": 1e-4,
    #"optim.weight_decay": 1e-4,
    #"optim.warmup_steps": 500,
    #"env.per_gpu_batch_size": 8,
    #"env.batch_size": 128,
    #"model.timm_image.checkpoint_name": "convnext_base",
}
#autoglue_predictor_lrp_img = autoglue_model_lrp_img.fit(train_lrp_img, hyperparameters=hyperparams)

In [232]:
#autoglue_predictor_lrp_img.fit_summary(verbosity=2)

In [233]:
#y_pred = autoglue_predictor_lrp_img.predict(X_test_img)
#record_regression_results(model_results, 'LRP', 'Autogluon_Mult', y_test_lrp, y_pred)


### Save predictions

In [234]:
test_inst = X_test.copy()
print(f"Number of test instances: {test_inst.shape[0]}")
metrics = ["iou", "lrp"]
models = ["baseline", "lr", "dt", "rf", "mlp", "xgb", "autg"]
for metric in metrics:
    for model in models:
        if metric == "iou":
            test_inst["GT"] = y_test
        else:
            test_inst["GT"] = y_test_lrp
        var_name = f"y_pred_{metric}_{model}"
        test_inst[model] = globals()[var_name]
    test_inst.to_csv(f"../results/assessors/{metric}_test_ass_" + data_path_file)


Number of test instances: 4137


# Final Model Comparison


In [235]:

results_df = pd.DataFrame(model_results, index=None)
results_df["test_r2"] = results_df["test_r2"].round(4)
results_df["test_mae"] = results_df["test_mae"].round(4)
#results_df["test_mse"] = results_df["test_mse"].round(4)
results_df.to_csv(f"../results/assessors/{model_det}_ass_results_table.csv")
display(results_df)


,target,model,test_r2,test_mae,test_ece
0,IoU,Constant Mean Predictor,-0.0064,0.1535,0.023651
1,IoU,Univariate Linear Regression (Confidence),0.3885,0.1181,0.012189
2,IoU,Linear Regression,0.4767,0.1090,0.011830
3,IoU,Decision Tree,0.4506,0.1109,0.010919
4,IoU,Random Forest,0.4869,0.1068,0.014599
5,IoU,MLP,0.4486,0.1099,0.007715
6,IoU,XGBoost,0.4853,0.1064,0.008797
7,IoU,Autogluon_Tab,0.4957,0.1066,0.012100
8,LRP,Constant Mean Predictor,-0.0059,0.1373,0.040283
9,LRP,Univariate Linear Regression (Confidence),0.4597,0.0989,0.017501


In [236]:
print(results_df.to_latex(index=False))

\begin{tabular}{llrrr}
\toprule
target & model & test_r2 & test_mae & test_ece \\
\midrule
IoU & Constant Mean Predictor & -0.006400 & 0.153500 & 0.023651 \\
IoU & Univariate Linear Regression (Confidence) & 0.388500 & 0.118100 & 0.012189 \\
IoU & Linear Regression & 0.476700 & 0.109000 & 0.011830 \\
IoU & Decision Tree & 0.450600 & 0.110900 & 0.010919 \\
IoU & Random Forest & 0.486900 & 0.106800 & 0.014599 \\
IoU & MLP & 0.448600 & 0.109900 & 0.007715 \\
IoU & XGBoost & 0.485300 & 0.106400 & 0.008797 \\
IoU & Autogluon_Tab & 0.495700 & 0.106600 & 0.012100 \\
LRP & Constant Mean Predictor & -0.005900 & 0.137300 & 0.040283 \\
LRP & Univariate Linear Regression (Confidence) & 0.459700 & 0.098900 & 0.017501 \\
LRP & Linear Regression & 0.518900 & 0.091500 & 0.015017 \\
LRP & Decision Tree & 0.517200 & 0.090700 & 0.018982 \\
LRP & Random Forest & 0.549300 & 0.087400 & 0.012271 \\
LRP & MLP & 0.509900 & 0.092500 & 0.017127 \\
LRP & XGBoost & 0.545700 & 0.087000 & 0.010825 \\
LRP & Autogluon